# 01 - Selection (KDD stage 1)
Choose and document the data. Target: flood risk for KZN and Mpumalanga districts.

| Source | Role | Status in this notebook |
|---|---|---|
| Sentinel-2 / Landsat 8 | spatial features (CNN) | SYNTHETIC tiles |
| SAWS stations | rainfall, river level, T, RH, pressure, wind (LSTM) | SYNTHETIC |
| PSMSL tide gauges | coastal sea level | not yet used |
| SANLC | land-cover statics | stand-in columns |
| EM-DAT / DesInventar | event labels | SYNTHETIC labels |

**Important:** results computed on synthetic data validate the pipeline only. See docs/DATA_SOURCES.md.

In [ ]:
{
 "cells": [
  {"cell_type": "markdown", "metadata": {}, "source": "# KDD Stage 1 — Selection\n\n**Real data sources this system is designed for** (see `docs/DATA_SOURCES.md` for URLs/licences):\n1. Sentinel-2 / Landsat 8 imagery (Copernicus Data Space, USGS, GEE)\n2. SAWS meteorological & hydrological stations\n3. PSMSL tide-gauge sea level\n4. SANLC land cover\n5. EM-DAT / DesInventar / gazette flood event records\n\n**This notebook does not download those sources** — no network access was available while building this prototype. It generates a **synthetic dataset with the same schema and units**, with flood signal deliberately embedded, so the rest of the KDD pipeline (02-05) and the live system can be demonstrated end-to-end offline. Replace the generator cells below with real API/download calls when connectivity and access approvals are in place; nothing downstream needs to change."},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "import numpy as np\nimport pandas as pd\nimport hashlib\nimport json\nfrom pathlib import Path\n\nSEED = 42\nrng = np.random.default_rng(SEED)\n\nREGIONS = [\n    (\"KZN-ETHEKWINI\", \"eThekwini (Durban)\", \"KwaZulu-Natal\"),\n    (\"KZN-UMGUNGUNDLOVU\", \"uMgungundlovu\", \"KwaZulu-Natal\"),\n    (\"KZN-KINGCETSHWAYO\", \"King Cetshwayo\", \"KwaZulu-Natal\"),\n    (\"MP-EHLANZENI\", \"Ehlanzeni\", \"Mpumalanga\"),\n    (\"MP-NKANGALA\", \"Nkangala\", \"Mpumalanga\"),\n]\nN_DAYS = 420\nTILE = 16\nDATASET_VERSION = \"synthetic-v1\"\n\nRAW = Path(\"../data/raw\")\nRAW.mkdir(parents=True, exist_ok=True)"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "def simulate_region(region_id: str, seed: int) -> pd.DataFrame:\n    \"\"\"Seasonal baseline + noise + injected flood episodes (SAWS-station-like schema).\"\"\"\n    r = np.random.default_rng(seed)\n    days = pd.date_range(\"2024-01-01\", periods=N_DAYS, freq=\"D\")\n    doy = days.dayofyear.values\n    season = 40 + 35 * np.sin(2 * np.pi * (doy - 60) / 365)  # summer-rainfall pattern\n    rainfall = np.clip(season * 0.4 + r.normal(0, 12, N_DAYS), 0, None)\n    river = 1.5 + rainfall.cumsum() * 0 * 0  # placeholder, replaced below via API index\n\n    label = np.zeros(N_DAYS, dtype=int)\n    n_events = r.integers(4, 8)\n    for _ in range(n_events):\n        start = r.integers(10, N_DAYS - 15)\n        dur = r.integers(3, 8)\n        rainfall[start:start + dur] += r.uniform(80, 220, dur)\n        label[start:start + dur] = 1\n\n    api = np.zeros(N_DAYS)  # antecedent precipitation index, decay factor 0.9\n    for i in range(1, N_DAYS):\n        api[i] = 0.9 * api[i - 1] + rainfall[i]\n    river_level = 0.6 + 0.02 * api + r.normal(0, 0.15, N_DAYS)\n    river_level = np.clip(river_level, 0, None)\n    soil_moisture = np.clip(0.25 + api / 400 + r.normal(0, 0.03, N_DAYS), 0, 1)\n    temperature = 22 - 6 * np.cos(2 * np.pi * doy / 365) + r.normal(0, 2, N_DAYS)\n    humidity = np.clip(60 + 20 * (rainfall > 20) + r.normal(0, 8, N_DAYS), 10, 100)\n\n    df = pd.DataFrame({\n        \"region_id\": region_id, \"date\": days, \"rainfall_mm_24h\": rainfall.round(1),\n        \"river_level_m\": river_level.round(2), \"soil_moisture\": soil_moisture.round(3),\n        \"temperature_c\": temperature.round(1), \"humidity_pct\": humidity.round(1),\n        \"flood_label\": label,\n    })\n    # missingness: simulate real station dropout, ~2% of rows, a short 3-day gap once\n    drop_idx = r.choice(N_DAYS, size=int(N_DAYS * 0.02), replace=False)\n    df.loc[drop_idx, [\"rainfall_mm_24h\", \"river_level_m\"]] = np.nan\n    gap_start = r.integers(50, N_DAYS - 60)\n    df.loc[gap_start:gap_start + 2, \"soil_moisture\"] = np.nan\n    return df"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "station_frames = [simulate_region(rid, seed=SEED + i) for i, (rid, _, _) in enumerate(REGIONS)]\nstations = pd.concat(station_frames, ignore_index=True)\nstations.to_csv(RAW / \"station_readings.csv\", index=False)\nstations.tail()"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "# SANLC-style static land cover composition per region (proportions sum to 1)\nland_cover = []\nfor rid, name, prov in REGIONS:\n    r = np.random.default_rng(hash(rid) % (2**32))\n    coastal = 1.0 if \"ETHEKWINI\" in rid or \"KINGCETSHWAYO\" in rid else 0.3\n    urban = r.uniform(0.15, 0.45) * (1.3 if \"ETHEKWINI\" in rid else 1.0)\n    water = r.uniform(0.02, 0.06) * (1.5 if coastal else 1.0)\n    veg = r.uniform(0.3, 0.55)\n    bare = max(0.0, 1 - urban - water - veg)\n    total = urban + water + veg + bare\n    land_cover.append({\"region_id\": rid, \"name\": name, \"province\": prov,\n                        \"urban_pct\": urban / total, \"water_pct\": water / total,\n                        \"vegetation_pct\": veg / total, \"bare_pct\": bare / total})\nland_cover_df = pd.DataFrame(land_cover)\nland_cover_df.to_csv(RAW / \"sanlc_land_cover.csv\", index=False)\nland_cover_df"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "# EM-DAT/DesInventar/gazette-style historical event log, derived from the labels above\nevents = []\nfor rid_df, (rid, _, _) in zip(station_frames, REGIONS):\n    lbl = rid_df[\"flood_label\"].values\n    in_event, start = False, None\n    for i, v in enumerate(lbl):\n        if v and not in_event:\n            in_event, start = True, rid_df[\"date\"].iloc[i]\n        if in_event and (not v or i == len(lbl) - 1):\n            end = rid_df[\"date\"].iloc[i - 1 if not v else i]\n            events.append({\"region_id\": rid, \"event_type\": \"flood\", \"start_date\": start, \"end_date\": end})\n            in_event = False\nevents_df = pd.DataFrame(events)\nevents_df.to_csv(RAW / \"historical_events.csv\", index=False)\nlen(events_df)"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "# Sentinel-2-style multispectral tiles (synthetic): a water-index channel brightens/grows\n# in a blob during labelled flood days, background texture otherwise. Shape per region: (N_DAYS, TILE, TILE, 4)\ndef make_tiles(label: np.ndarray, seed: int) -> np.ndarray:\n    r = np.random.default_rng(seed)\n    tiles = r.normal(0.2, 0.05, (len(label), TILE, TILE, 4)).astype(\"float32\")\n    yy, xx = np.mgrid[0:TILE, 0:TILE]\n    cy, cx = TILE / 2, TILE / 2\n    for i, lab in enumerate(label):\n        if lab:\n            radius = r.uniform(3, 7)\n            blob = (((yy - cy) ** 2 + (xx - cx) ** 2) < radius ** 2).astype(\"float32\")\n            tiles[i, :, :, 3] += blob * r.uniform(0.4, 0.8)  # NDWI-like water channel\n    return np.clip(tiles, 0, 1)\n\ntile_arrays = {rid: make_tiles(df[\"flood_label\"].values, SEED + i)\n               for i, (df, (rid, _, _)) in enumerate(zip(station_frames, REGIONS))}\nnp.savez_compressed(RAW / \"satellite_tiles.npz\", **tile_arrays)\n{k: v.shape for k, v in tile_arrays.items()}"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "# Data integrity: SHA-256 checksums for every raw artefact (spec 4.1)\ndef sha256_of(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(8192), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\nchecksums = {p.name: sha256_of(p) for p in RAW.glob(\"*\") if p.is_file()}\n(RAW / \"CHECKSUMS.sha256.json\").write_text(json.dumps(checksums, indent=2))\n(RAW / \"DATASET_VERSION.txt\").write_text(DATASET_VERSION)\nchecksums"}
 ],
 "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python", "version": "3.10"}},
 "nbformat": 4,
 "nbformat_minor": 5
}

In [ ]:
import sys
from pathlib import Path

try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "common.py").exists())
except StopIteration:
    raise SystemExit("notebooks/common.py not found. Run inside the repo (on Colab: clone it first).")
sys.path.insert(0, str(ROOT / "notebooks"))

import numpy as np
import pandas as pd
from common import REGIONS, SEED, generate_synthetic, set_seed, sha256_file, write_meta

set_seed()
(ROOT / "data").mkdir(exist_ok=True)

In [ ]:
raw = ROOT / "data" / "raw"
real_files = sorted(raw.glob("*")) if raw.exists() else []
print("Real raw files found:", [f.name for f in real_files] or "none")
USE_SYNTHETIC = True  # set False only after real loaders are written below
if not USE_SYNTHETIC:
    raise NotImplementedError("Write loaders for the files in data/raw (see docs/DATA_SOURCES.md).")

In [ ]:
df, tiles = generate_synthetic(years=4, seed=SEED)
csv_path = ROOT / "data" / "ewsa_selected.csv"
tile_path = ROOT / "data" / "ewsa_tiles.npz"
df.to_csv(csv_path, index=False)
np.savez_compressed(tile_path, tiles=tiles)
write_meta(
    ROOT / "data" / "ewsa_selected.meta.json",
    data_kind="synthetic", seed=SEED, rows=len(df), regions=list(REGIONS),
    sha256={csv_path.name: sha256_file(csv_path), tile_path.name: sha256_file(tile_path)},
)
print(df.shape, tiles.shape)
print("Flood-day prevalence:", round(float(df["event"].mean()), 4))
assert 0.005 < df["event"].mean() < 0.30, "prevalence outside sane range"